# 02 – Silver ➜ Gold star schema (ProductIQ)

Builds the **gold** Delta tables used by the *ProductIQ Data Agent* (Fabric data agent) and by the Foundry agents through
Fabric IQ. Column names are friendly `snake_case` so the data agent can map business questions to SQL.

| Table | Grain |
|---|---|
| `dim_date` | one row per calendar day 2024-11-01 … 2026-10-31 |
| `dim_product`, `dim_component`, `dim_supplier`, `dim_plant`, `dim_region` | one row per master-data entity |
| `fact_feedback` | one review or support ticket |
| `fact_sales` | product × region × channel × day |
| `fact_bom_cost` | product × component × month (BOM version valid in that month) |
| `fact_supplier_capacity` | supplier × component × month |
| `fact_logistics` | one shipment |
| `fact_market_signal` | month × region × category × metric |
| `fact_warranty` | one warranty claim |

Run `01_bronze_to_silver` first. Deterministic and idempotent (`overwrite`). The pandas mirror of this logic lives in
`data/generator/reference_pipeline.py` and is used to compute the evaluation answers in `eval/golden_qa.jsonl`.

In [ ]:
import datetime as dt
from pyspark.sql import functions as F
from pyspark.sql import Row, Window

WINDOW_START = "2024-11-01"
WINDOW_END = "2026-10-31"


def write_table(df, name: str):
    df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(name)
    print(f"{name:<28}{spark.table(name).count():>12,} rows")


def ym(c):
    return F.date_format(F.col(c), "yyyy-MM")


def dkey(c):
    return F.date_format(F.col(c), "yyyyMMdd").cast("int")

## dim_date (retail event calendar, fiscal year starting in April)

In [ ]:
def season_event(d: dt.date) -> str:
    """Retail event calendar used by dim_date (identical code in notebook 02)."""
    nov1 = dt.date(d.year, 11, 1)
    thanksgiving = nov1 + dt.timedelta(days=(3 - nov1.weekday()) % 7 + 21)
    black_friday = thanksgiving + dt.timedelta(days=1)
    if black_friday <= d <= black_friday + dt.timedelta(days=3):
        return "black_friday"
    if d.month == 11 and d.day <= 11:
        return "singles_day"
    if d.month == 12 and d.day >= 15:
        return "year_end"
    if (d.month == 4 and d.day >= 29) or (d.month == 5 and d.day <= 5):
        return "golden_week"
    if d.month in (7, 8):
        return "summer"
    return "none"


def build_dim_date_rows(start: str = WINDOW_START, end: str = WINDOW_END) -> list[dict]:
    d0 = dt.date.fromisoformat(start)
    d1 = dt.date.fromisoformat(end)
    rows = []
    d = d0
    while d <= d1:
        fy = d.year if d.month >= 4 else d.year - 1
        fq = ((d.month - 4) % 12) // 3 + 1
        rows.append({
            "date_key": int(d.strftime("%Y%m%d")), "date": d.isoformat(), "year": d.year,
            "quarter": (d.month - 1) // 3 + 1, "month": d.month, "month_name": d.strftime("%B"),
            "year_month": d.strftime("%Y-%m"), "week_of_year": d.isocalendar()[1],
            "day_of_week": d.isoweekday(), "day_name": d.strftime("%A"),
            "is_weekend": d.isoweekday() >= 6, "fiscal_year": f"FY{fy}", "fiscal_quarter": f"FY{fy}-Q{fq}",
            "season_event": season_event(d),
        })
        d += dt.timedelta(days=1)
    return rows


dim_date = (spark.createDataFrame([Row(**r) for r in build_dim_date_rows()])
            .withColumn("date", F.to_date("date"))
            .select("date_key", "date", "year", "quarter", "month", "month_name", "year_month", "week_of_year",
                    "day_of_week", "day_name", "is_weekend", "fiscal_year", "fiscal_quarter", "season_event"))
write_table(dim_date, "dim_date")

## Dimensions

In [ ]:
p = spark.table("silver_products")
dim_product = p.select(
    F.col("product_code").alias("product_id"), "product_name", "category", "subcategory", "series", "launch_date",
    "discontinue_date",
    (F.col("discontinue_date").isNull() | (F.col("discontinue_date") > F.lit(WINDOW_END).cast("date"))).alias("is_active"),
    "list_price_usd", "target_segment", F.col("primary_plant_code").alias("primary_plant_id"), "has_lidar",
    "has_self_empty_dock", "has_mop", "is_pet_edition")
write_table(dim_product, "dim_product")

dim_component = spark.table("silver_components").select(
    F.col("component_code").alias("component_id"), "component_name", "component_category", "unit_of_measure",
    "standard_cost_usd", "is_critical", "recyclable_pct", "weight_g", "primary_supplier_id")
write_table(dim_component, "dim_component")

dim_supplier = spark.table("silver_suppliers").select(
    "supplier_id", "supplier_name", "country", F.col("tier").alias("supplier_tier"), "supply_category",
    "quality_rating", "esg_score", "status", "onboarded_date")
write_table(dim_supplier, "dim_supplier")

dim_plant = spark.table("silver_plants").select(
    F.col("plant_code").alias("plant_id"), "plant_name", "plant_type", "country", "city", "monthly_capacity_units",
    "opened_year")
write_table(dim_plant, "dim_plant")

dim_region = spark.table("silver_regions").select(
    "region_code", "region_name", "country", "market_cluster", "currency", "fx_to_usd")
write_table(dim_region, "dim_region")

## fact_feedback (reviews + tickets)

In [ ]:
FEEDBACK_COLUMNS = ["feedback_id", "source", "feedback_date", "date_key", "year_month", "product_id", "region_code",
                    "customer_id", "channel", "language", "rating", "is_pet_owner", "pet_owner_source", "topic",
                    "sentiment_score", "sentiment_label", "is_negative", "title", "feedback_text", "verified_purchase",
                    "helpful_votes", "ticket_priority", "ticket_status", "csat_score"]

rev = spark.table("silver_reviews").select(
    F.col("review_id").alias("feedback_id"), F.lit("review").alias("source"), F.col("review_date").alias("feedback_date"),
    F.col("product_code").alias("product_id"), "region_code", "customer_id", F.col("platform").alias("channel"),
    "language", F.col("rating").cast("int").alias("rating"), F.col("pet_owner").alias("is_pet_owner"), "pet_owner_source",
    "topic", "sentiment_score", "sentiment_label", "title", F.col("review_text").alias("feedback_text"),
    "verified_purchase", F.col("helpful_votes").cast("int").alias("helpful_votes"),
    F.lit(None).cast("string").alias("ticket_priority"), F.lit(None).cast("string").alias("ticket_status"),
    F.lit(None).cast("int").alias("csat_score"))

tck = spark.table("silver_tickets").select(
    F.col("ticket_id").alias("feedback_id"), F.lit("ticket").alias("source"), F.to_date("created_at").alias("feedback_date"),
    F.col("product_code").alias("product_id"), "region_code", "customer_id", F.col("contact_channel").alias("channel"),
    "language", F.lit(None).cast("int").alias("rating"), F.col("pet_owner").alias("is_pet_owner"), "pet_owner_source",
    "topic", "sentiment_score", "sentiment_label", F.col("subject").alias("title"),
    F.col("description").alias("feedback_text"), F.lit(None).cast("boolean").alias("verified_purchase"),
    F.lit(None).cast("int").alias("helpful_votes"), F.col("priority").alias("ticket_priority"),
    F.col("status").alias("ticket_status"), F.col("csat_score").cast("int").alias("csat_score"))

fact_feedback = (rev.unionByName(tck)
                 .withColumn("date_key", dkey("feedback_date"))
                 .withColumn("year_month", ym("feedback_date"))
                 .withColumn("is_negative", F.col("sentiment_label") == F.lit("negative"))
                 .select(*FEEDBACK_COLUMNS))
write_table(fact_feedback, "fact_feedback")

## fact_sales

In [ ]:
fact_sales = spark.table("silver_sales").select(
    "order_line_id", F.col("order_date").alias("sales_date"), dkey("order_date").alias("date_key"),
    ym("order_date").alias("year_month"), F.col("product_code").alias("product_id"), "region_code", "channel",
    F.col("plant_code").alias("plant_id"), F.col("units").alias("units_sold"), "unit_price_local", "currency",
    "gross_revenue_usd", "discount_pct", "net_revenue_usd")
write_table(fact_sales, "fact_sales")

## fact_bom_cost (product × component × month, BOM version valid in the month)

In [ ]:
months = (spark.sql(f"SELECT explode(sequence(to_date('{WINDOW_START}'), to_date('2026-10-01'), interval 1 month)) AS month")
          .withColumn("month_end", F.last_day("month")))
prod_life = spark.table("silver_products").select(F.col("product_code").alias("_p"), "launch_date", "discontinue_date")
bom = spark.table("silver_bom").join(prod_life, F.col("product_code") == F.col("_p"), "inner").drop("_p")
costs = spark.table("silver_component_costs").select(
    F.col("month").alias("_cm"), F.col("component_code").alias("_cc"), "unit_cost_usd", "primary_supplier_id")

fact_bom_cost = (bom.crossJoin(months)
                 .filter((F.col("effective_from") <= F.col("month"))
                         & (F.col("effective_to").isNull() | (F.col("effective_to") >= F.col("month")))
                         & (F.col("launch_date") <= F.col("month_end"))
                         & (F.col("discontinue_date").isNull() | (F.col("discontinue_date") >= F.col("month"))))
                 .join(costs, (F.col("month") == F.col("_cm")) & (F.col("component_code") == F.col("_cc")), "left")
                 .select("month", ym("month").alias("year_month"), F.col("product_code").alias("product_id"),
                         F.col("component_code").alias("component_id"), F.col("primary_supplier_id").alias("supplier_id"),
                         "bom_version", F.col("quantity").alias("quantity_per_unit"), "unit_cost_usd",
                         F.round(F.col("quantity") * F.col("unit_cost_usd"), 4).alias("extended_cost_usd")))
write_table(fact_bom_cost, "fact_bom_cost")

## fact_supplier_capacity (supplier count per component and month, single-source flag, risk level)

In [ ]:
w = Window.partitionBy("month", "component_code")
cap = (spark.table("silver_supplier_capacity")
       .withColumn("supplier_count_for_component",
                   F.sum(F.when(F.coalesce(F.col("committed_units"), F.lit(0)) > 0, 1).otherwise(0)).over(w).cast("int"))
       .withColumn("is_single_source", F.col("supplier_count_for_component") == 1))
u = F.col("utilisation_pct")
cap = cap.withColumn("risk_level",
                     F.when((u >= 90) | (F.col("is_single_source") & (u >= 80)), "high")
                      .when((u >= 75) | F.col("is_single_source"), "medium")
                      .otherwise("low"))
fact_supplier_capacity = cap.select(
    "month", ym("month").alias("year_month"), "supplier_id", F.col("component_code").alias("component_id"),
    "capacity_units", "committed_units", "utilisation_pct", "lead_time_days", "alt_supplier_qualification_days",
    "alt_supplier_candidate_id", "supplier_count_for_component", "is_single_source", "risk_level")
write_table(fact_supplier_capacity, "fact_supplier_capacity")

## fact_logistics

In [ ]:
sh = spark.table("silver_shipments").withColumn("_transit", F.datediff("delivery_date", "ship_date"))
fact_logistics = sh.select(
    "shipment_id", "ship_date", dkey("ship_date").alias("date_key"), ym("ship_date").alias("year_month"),
    "delivery_date", F.col("origin_plant_code").alias("origin_plant_id"), "destination_region_code",
    F.col("product_code").alias("product_id"), "transport_mode", "carrier_name", "units", "weight_kg", "distance_km",
    "freight_cost_usd",
    F.round(F.col("freight_cost_usd") / F.col("weight_kg"), 4).alias("freight_cost_per_kg_usd"),
    F.round(F.col("freight_cost_usd") / F.col("units"), 4).alias("freight_cost_per_unit_usd"),
    "co2e_kg", "planned_transit_days", F.col("_transit").alias("actual_transit_days"),
    (F.col("_transit") - F.col("planned_transit_days")).alias("delay_days"),
    F.when(F.col("_transit").isNull(), F.lit(None).cast("boolean"))
     .otherwise(F.col("_transit") <= F.col("planned_transit_days")).alias("is_on_time"),
    F.when(F.col("delivery_date").isNull(), "in_transit").otherwise("delivered").alias("shipment_status"))
write_table(fact_logistics, "fact_logistics")

## fact_market_signal and fact_warranty

In [ ]:
fact_market_signal = spark.table("silver_market_signals").select(
    "month", ym("month").alias("year_month"), "region_code", "category", "metric", "value", "unit", "source")
write_table(fact_market_signal, "fact_market_signal")

fact_warranty = spark.table("silver_warranty").select(
    "claim_id", "claim_date", dkey("claim_date").alias("date_key"), ym("claim_date").alias("year_month"),
    F.col("product_code").alias("product_id"), "region_code", "customer_id", F.col("pet_owner").alias("is_pet_owner"),
    F.col("failed_component_code").alias("component_id"), "failure_mode", "repair_action", "claim_status",
    "claim_cost_usd", "purchase_date", "days_in_service")
write_table(fact_warranty, "fact_warranty")

## Row counts

In [ ]:
GOLD_TABLES = ["dim_product", "dim_component", "dim_supplier", "dim_plant", "dim_region", "dim_date",
               "fact_feedback", "fact_sales", "fact_bom_cost", "fact_supplier_capacity", "fact_logistics",
               "fact_market_signal", "fact_warranty"]
for t in GOLD_TABLES:
    print(f"{t:<28}{spark.table(t).count():>12,}")